# UniGeoCLIP demo

End-to-end on **synthetic data, CPU**: generate fake data for all five
modalities, train the multimodal contrastive model for a few steps, then run the
three downstream evals (pv4ger classification, chesapeake segmentation, PDFM
regression).

> Run this notebook from the **repository root** (the parent of `unigeoclip/`),
> with the deps from `requirements.txt` installed (including `scenic` and
> `big_vision`). Two CPU devices are not required; one is fine for the demo.

## 0. Setup

In [ ]:
import os
import tempfile

import jax
import ml_collections
import numpy as np

print("JAX devices:", jax.devices())

## 1. Generate fake data

`generate_fake_downstream_data` writes the three downstream benchmarks; the
multimodal generator writes the 5-modality training shards (sv / aerial / dsm /
text / latlng).

In [ ]:
from unigeoclip.data import generate_fake_downstream_data as gen_ds
from unigeoclip.data.generate_fake_multimodal_data import write_multimodal_dataset

DATA = tempfile.mkdtemp(prefix="unigeoclip_demo_")
rng = np.random.default_rng(0)

gen_ds._write_pv4ger(DATA, image_size=64, rng=rng)
gen_ds._write_chesapeake(DATA, size=32, num_classes=7, rng=rng)
gen_ds._write_pdfm(DATA, num_regions=120, num_targets=4, rng=rng)
write_multimodal_dataset(
    os.path.join(DATA, "area"), num_train=16, num_valid=8,
    sv_image_size=64, aerial_image_size=64, dsm_image_size=64, text_dim=32,
)
print("data root:", DATA)
print("contents:", sorted(os.listdir(DATA)))

## 2. Train the 5-modality contrastive model (a few steps)

We use the `tiny` preset and point the data config at the generated multimodal
shards (`--config.data.location`). Five steps is enough to see the InfoNCE
losses over every modality pair.

In [ ]:
from clu import metric_writers
from unigeoclip.configs import contrastive_merge as cfgmod
from unigeoclip import datasets, load, trainer

cfg = cfgmod.get_config("tiny")
cfg.device_count = jax.device_count()
cfg.num_training_steps = 5
cfg.batch_size_per_device = 2
cfg.checkpoint = False
cfg.data.data_dir = DATA
cfg.data.location = "area"          # read the generated shards
cfg.data.text_dim = 32
cfg = cfg.copy_and_resolve_references()

WORKDIR = tempfile.mkdtemp(prefix="unigeoclip_run_")
ds = datasets.build_dataset(cfg.data, cfg.batch_size, cfg.batch_size_eval, cfg.rng_seed)
model = load.get_model(cfg.model_name)(cfg.model, ds.meta_data)
exp = trainer.Experiment.init(
    rng=jax.random.PRNGKey(0), config=cfg, model=model,
    dummy_input=ds.meta_data["dummy_batch_fn"](),
)
writer = metric_writers.create_default_writer(WORKDIR, just_logging=True)
train_summary, _ = exp.train_loop(ds, writer, workdir=None)
print("modalities:", model.modalities)
print("InfoNCE losses:",
      {k: round(float(v), 3) for k, v in train_summary.items() if "loss/" in k})

## 3. Downstream evals

The evals pull features from a model through the encoder interface
(`get_aerial_embeddings`, `get_dense_aerial_embeddings`,
`get_location_embeddings`). **pv4ger** and **PDFM** run against the
`contrastive_merge` model we just trained (its aerial and `latlng` towers).
**chesapeake**'s DPT head needs *multi-depth* dense features, which the SigLIP
aerial backbone provides via `encode_dense(get_depths=...)` (the in-repo ViT's
dense path is single-depth), so we build a small `contrastive_merge` variant
with a SigLIP aerial tower for it.

In [ ]:
def min_train_config(model_cfg, model_name):
  c = ml_collections.ConfigDict()
  c.device_count = jax.device_count()
  c.batch_size = 2
  c.model = model_cfg
  c.model_name = model_name
  c.lr = ml_collections.ConfigDict(dict(
      learning_rate_schedule="compound", factors="constant",
      base_learning_rate=1e-3, start_decay_step=0, steps_per_cycle=100))
  c.optimizer = ml_collections.ConfigDict(dict(
      optax_name="scale_by_adam",
      optax_configs=ml_collections.ConfigDict(dict(b1=0.9, b2=0.95)),
      weight_decay=0.0))
  c.rate_fns = {}
  c.max_grad_norm = None
  c.debug_train = c.debug_eval = False
  c.avoid_weight_decay_on_frozen_params = False
  return c


def image_eval_config(name, split, num_classes):
  c = ml_collections.ConfigDict()
  c.batch_size = c.batch_size_eval = 2
  c.log_progress_interval = 100
  c.num_classes = num_classes
  c.batch_sizes_eval = [2]
  c.learning_rates = [1e-3]
  c.n_epochs = 2          # tiny: keep the probe fast
  c.k_values = (1,)
  c.data = ml_collections.ConfigDict()
  c.data.dataset_name = name
  c.data.data_dir = os.path.join(DATA, name)
  c.data.train_split = "train"
  c.data.valid_split = split
  c.data.shuffle_train = c.data.shuffle_valid = False
  c.data.grain_num_workers = c.data.grain_num_workers_valid = 0
  c.data.grain_num_threads = c.data.grain_num_threads_valid = 1
  c.data.grain_prefetch_buffer_size = 1
  c.data.preprocess = ml_collections.ConfigDict()
  return c

### 3a. pv4ger — classification (k-NN + linear probe)

In [ ]:
from unigeoclip.evals.classif_eval import ClassificationEvaluation

# pv4ger only needs pooled aerial embeddings, so we reuse the trained model
# directly (its ViT aerial tower implements get_aerial_embeddings).
_, pv4ger_metrics, _ = ClassificationEvaluation(
    model, image_eval_config("pv4ger_classification", "valid", 2)
).evaluate(exp.train_state)
print("pv4ger:", pv4ger_metrics)

### 3b. chesapeake — segmentation (DPT head on frozen features)

> This cell trains a small DPT head, so it takes ~1–2 min on CPU.

In [ ]:
from unigeoclip.data.multimodal_dummy_source import MultiModalDummyDataSource
from unigeoclip.configs import defaults
from unigeoclip.evals.semseg_eval import SegmentationEvaluation

# chesapeake's DPT head needs multi-depth dense features, which the SigLIP aerial
# backbone exposes via encode_dense(get_depths=...) (the in-repo ViT's dense path
# is single-depth). We build a small contrastive_merge with a SigLIP aerial tower
# -- randomly initialised (load_pretrained=False, so no SigLIP2 checkpoint is
# needed) and initialised at the 32px chesapeake tile size.
ches_model_cfg = cfgmod.model_config("tiny", ["sv", "aerial"])
siglip_cfg = defaults.siglip(version="Ti", patch=16).unlock()
siglip_cfg.model_name = "SigLIP2 Ti/16 256"
siglip_cfg.load_pretrained = False
ches_model_cfg.aerial_encoder = siglip_cfg

src32 = MultiModalDummyDataSource(
    num_examples=8, sv_image_size=32, aerial_image_size=32,
    dsm_image_size=32, text_dim=32)
ches_batch = jax.tree.map(lambda *xs: np.stack(xs), *[src32[i] for i in range(2)])

ches_model = load.get_model("contrastive_merge")(ches_model_cfg, {})
ches_exp = trainer.Experiment.init(
    rng=jax.random.PRNGKey(0),
    config=min_train_config(ches_model_cfg, "contrastive_merge"),
    model=ches_model,
    dummy_input=ches_batch,
)

_, chesapeake_metrics, _ = SegmentationEvaluation(
    ches_model, image_eval_config("m-chesapeake", "test", 7)
).evaluate(ches_exp.train_state)
print("chesapeake:", chesapeake_metrics)

### 3c. PDFM — location-embedding regression (mean test R²)

In [ ]:
from unigeoclip.evals.pdfm_eval import PDFMEvaluation

pdfm_cfg = ml_collections.ConfigDict()
pdfm_cfg.batch_size_eval = 16
pdfm_cfg.data = ml_collections.ConfigDict()
pdfm_cfg.data.data_dir = os.path.join(DATA, "pdfm")

_, pdfm_metrics, _ = PDFMEvaluation(model, pdfm_cfg).evaluate(exp.train_state)
print("pdfm:", pdfm_metrics)

## 4. Next steps

- Scale up: `--config=...:B` and more `--config.num_training_steps`.
- Train from real data: implement a `grain.RandomAccessDataSource` with the same
  fields (see `data/multimodal_dummy_source.py` / `data/spec.py`).
- Evaluate your own encoder: implement the encoder methods on a `BaseModel`
  subclass (`models/contrastive_merge.py` is the reference
  implementation) and register it in `load.py`. See `unigeoclip/README.md`.